# 点金术基金化策略回测与宏观对比分析系统

## 你遇到的报错意味着什么？
`No fund data loaded successfully` 通常不是策略代码问题，而是**数据获取阶段**失败：
- 网络/API暂时不可用
- 标的代码有误或接口字段变动
- 本地无缓存且首次拉取失败

本 Notebook 已重构为：
1. **自动增量下载**（优先在线更新，失败自动回退本地缓存）
2. **模块化函数**（数据层 / 策略层 / 评估层 / 可视化层）
3. **分红处理优先全收益口径**（`adjusted_price`）
4. 新增对比策略：**买入并持有（分红再投资）**

---

## 推荐项目结构（自动创建）

```text
ma120/
  ma120_backtest.ipynb
  data/
    raw/
      funds/      # 每只基金一个 parquet，自动增量更新
      macro/      # CPI / 国债收益率缓存
    processed/    # 合并后结果、指标输出（可扩展）
```

只需维护 `params['fund_pool']`，系统会按标的自动更新历史数据。

In [ ]:
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import akshare as ak
except ImportError as exc:
    raise ImportError("请先安装 akshare: pip install akshare") from exc


# ============================================================
# A. 参数配置（唯一需要常改的入口）
# ============================================================
params = {
    "ma_window": 120,
    "buy_threshold": 0.95,
    "sell_threshold": 1.05,
    "start_date": "2015-03-17",
    "end_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
    "initial_cash": 1_000_000,
    "fund_pool": [
        "510880", "515180", "512890", "515080","159307",
        "515100", "515300", "560020", "515450", "512530",
        "563020","510720","561580","513820"
    ],#"159905",
    "benchmark_hold_symbol": "510880",  # 买入并持有对比基准
}
# 
PROJECT_ROOT = Path(".").resolve()
RAW_FUNDS_DIR = PROJECT_ROOT / "data" / "raw" / "funds"
RAW_MACRO_DIR = PROJECT_ROOT / "data" / "raw" / "macro"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
for p in [RAW_FUNDS_DIR, RAW_MACRO_DIR, PROCESSED_DIR]:
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# B. 数据层：基金（自动增量下载 + 本地缓存回退）
# ============================================================
def _find_col(columns: List[str], keys: List[str]) -> Optional[str]:
    mapping = {str(c).lower(): c for c in columns}
    for k in keys:
        k = k.lower()
        for low, raw in mapping.items():
            if k in low:
                return raw
    return None


def _manual_total_return_adjust(close: pd.Series, dividend_cash: pd.Series) -> pd.Series:
    div = pd.to_numeric(dividend_cash, errors="coerce").fillna(0.0)
    prev_close = close.shift(1).replace(0, np.nan)
    reinvest_factor = (1.0 + (div / prev_close).fillna(0.0)).cumprod()
    return close * reinvest_factor


def _normalize_fund_df(raw: pd.DataFrame, symbol: str, adjust_mode: str) -> pd.DataFrame:
    if raw is None or raw.empty:
        raise ValueError(f"{symbol}: 空数据")

    cols = list(raw.columns)
    date_col = _find_col(cols, ["日期", "date", "day"])
    close_col = _find_col(cols, ["收盘", "close", "单位净值"])
    if date_col is None or close_col is None:
        raise ValueError(f"{symbol}: 无法识别日期/收盘字段，列={cols}")

    out = pd.DataFrame()
    out["date"] = pd.to_datetime(raw[date_col], errors="coerce")
    out["close"] = pd.to_numeric(raw[close_col], errors="coerce")

    # 优先使用全收益/复权字段
    adjusted_col = _find_col(cols, ["累计净值", "复权", "adjusted", "adj"])
    if adjusted_col is not None:
        out["adjusted_price"] = pd.to_numeric(raw[adjusted_col], errors="coerce")
        adjust_source = f"vendor_adjusted:{adjusted_col}"
    else:
        # 若接口使用了 qfq/hfq，可认为 close 已按该口径处理
        if adjust_mode in ["qfq", "hfq"]:
            out["adjusted_price"] = out["close"]
            adjust_source = f"ak_adjust_mode:{adjust_mode}"
        else:
            # fallback: 手动分红复权（若字段可用）
            div_col = _find_col(cols, ["分红", "dividend", "派息", "现金红利"])
            if div_col is not None:
                out["adjusted_price"] = _manual_total_return_adjust(
                    out["close"], pd.to_numeric(raw[div_col], errors="coerce")
                )
                adjust_source = f"manual_dividend_fallback:{div_col}"
            else:
                out["adjusted_price"] = out["close"]
                adjust_source = "close_proxy_warning"
                warnings.warn(
                    f"{symbol}: 未找到累计净值/分红字段，adjusted_price退化为close，可能存在除息假信号。"
                )

    out = out.dropna(subset=["date", "close", "adjusted_price"]).drop_duplicates(subset=["date"])
    out = out.sort_values("date").set_index("date")
    out["adjust_source"] = adjust_source
    return out


def _fetch_fund_from_ak(symbol: str, start_date: str, end_date: str) -> Tuple[pd.DataFrame, str]:
    start_s = pd.Timestamp(start_date).strftime("%Y%m%d")
    end_s = pd.Timestamp(end_date).strftime("%Y%m%d")

    last_error = None
    # 多模式尝试，提升接口兼容性
    for adjust_mode in ["hfq", "qfq", ""]:
        try:
            raw = ak.fund_etf_hist_em(
                symbol=symbol,
                period="daily",
                start_date=start_s,
                end_date=end_s,
                adjust=adjust_mode,
            )
            norm = _normalize_fund_df(raw, symbol=symbol, adjust_mode=adjust_mode)
            if not norm.empty:
                return norm, adjust_mode
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"akshare 拉取失败: {last_error}")


def _fund_cache_path(symbol: str) -> Path:
    return RAW_FUNDS_DIR / f"{symbol}.parquet"


def _read_cached_fund(symbol: str) -> Optional[pd.DataFrame]:
    fp = _fund_cache_path(symbol)
    if not fp.exists():
        return None
    df = pd.read_parquet(fp)
    df.index = pd.to_datetime(df.index)
    return df.sort_index()


def _write_cached_fund(symbol: str, df: pd.DataFrame) -> None:
    df.sort_index().to_parquet(_fund_cache_path(symbol))


def load_or_update_one_fund(symbol: str, start_date: str, end_date: str) -> Tuple[Optional[pd.DataFrame], dict]:
    cached = _read_cached_fund(symbol)
    log = {
        "symbol": symbol,
        "status": "",
        "source": "",
        "rows": 0,
        "cache_used": False,
        "updated": False,
    }

    if cached is None or cached.empty:
        # 首次全量拉取
        try:
            fresh, mode = _fetch_fund_from_ak(symbol, start_date, end_date)
            _write_cached_fund(symbol, fresh)
            log.update({"status": "ok", "source": f"online_full:{mode}", "rows": len(fresh), "updated": True})
            return fresh, log
        except Exception as exc:
            log.update({"status": f"failed:{exc}", "source": "none", "rows": 0})
            return None, log

    # 有缓存时：增量更新
    cached = cached.sort_index()
    log["cache_used"] = True
    last_date = cached.index.max()
    next_date = (last_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    if pd.Timestamp(next_date) <= pd.Timestamp(end_date):
        try:
            inc, mode = _fetch_fund_from_ak(symbol, next_date, end_date)
            merged = pd.concat([cached, inc], axis=0)
            merged = merged[~merged.index.duplicated(keep="last")].sort_index()
            _write_cached_fund(symbol, merged)
            out = merged.loc[(merged.index >= pd.Timestamp(start_date)) & (merged.index <= pd.Timestamp(end_date))]
            log.update({"status": "ok", "source": f"cache+increment:{mode}", "rows": len(out), "updated": True})
            return out, log
        except Exception as exc:
            # 在线增量失败，回退缓存
            out = cached.loc[(cached.index >= pd.Timestamp(start_date)) & (cached.index <= pd.Timestamp(end_date))]
            log.update({"status": f"cache_only:{exc}", "source": "cache", "rows": len(out), "updated": False})
            return out if len(out) > 0 else None, log

    out = cached.loc[(cached.index >= pd.Timestamp(start_date)) & (cached.index <= pd.Timestamp(end_date))]
    log.update({"status": "ok", "source": "cache_no_update_needed", "rows": len(out), "updated": False})
    return out if len(out) > 0 else None, log


def load_fund_data(symbols: List[str], start_date: str, end_date: str) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    data: Dict[str, pd.DataFrame] = {}
    logs = []
    all_dates = set()

    for symbol in symbols:
        df, log = load_or_update_one_fund(symbol, start_date, end_date)
        logs.append(log)
        if df is not None and not df.empty:
            data[symbol] = df
            all_dates.update(df.index.tolist())

    log_df = pd.DataFrame(logs)
    if not data:
        msg = (
            "No fund data loaded.\n"
            "请先检查：\n"
            "1) 网络是否可访问 akshare 数据源；\n"
            "2) 基金代码是否正确；\n"
            "3) 若外网受限，可先手动放入 data/raw/funds/{symbol}.parquet（字段至少包含 close, adjusted_price）。"
        )
        raise RuntimeError(msg)

    # 统一交易日历 + 停牌缺失前向填充
    aligned_dates = pd.DatetimeIndex(sorted(all_dates))
    for symbol, df in data.items():
        x = df.reindex(aligned_dates).sort_index()
        x["close"] = x["close"].ffill()
        x["adjusted_price"] = x["adjusted_price"].ffill()
        x["adjust_source"] = x["adjust_source"].ffill().fillna("unknown")
        data[symbol] = x

    return data, log_df


# ============================================================
# C. 宏观数据层：CPI + 国债收益率（带缓存回退）
# ============================================================
def _read_macro_cache(name: str) -> Optional[pd.Series]:
    fp = RAW_MACRO_DIR / f"{name}.csv"
    if not fp.exists():
        return None
    temp = pd.read_csv(fp)
    temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
    temp = temp.dropna().drop_duplicates(subset=["date"]).set_index("date").sort_index()
    return pd.to_numeric(temp[name], errors="coerce").rename(name)


def _write_macro_cache(name: str, s: pd.Series) -> None:
    df = s.rename(name).reset_index().rename(columns={"index": "date"})
    df.to_csv(RAW_MACRO_DIR / f"{name}.csv", index=False)


def load_cpi_data(start_date: str, end_date: str) -> pd.Series:
    cpi = None
    for fname in ["macro_china_cpi", "macro_china_cpi_monthly"]:
        if hasattr(ak, fname):
            try:
                raw = getattr(ak, fname)()
                dcol = _find_col(list(raw.columns), ["日期", "date", "月份", "month"])
                vcol = _find_col(list(raw.columns), ["同比", "yoy", "cpi"])
                if dcol and vcol:
                    t = raw[[dcol, vcol]].copy()
                    t.columns = ["date", "inflation_rate"]
                    t["date"] = pd.to_datetime(t["date"], errors="coerce")
                    t["inflation_rate"] = pd.to_numeric(t["inflation_rate"], errors="coerce")
                    cpi = t.dropna().drop_duplicates(subset=["date"]).set_index("date").sort_index()["inflation_rate"]
                    break
            except Exception:
                continue

    if cpi is None:
        cpi = _read_macro_cache("inflation_rate")
        if cpi is None:
            raise RuntimeError("CPI在线与缓存均不可用。可手工保存 data/raw/macro/inflation_rate.csv")
    else:
        _write_macro_cache("inflation_rate", cpi)

    if cpi.dropna().median() > 1:
        cpi = cpi / 100.0
    return cpi.loc[(cpi.index >= pd.Timestamp(start_date)) & (cpi.index <= pd.Timestamp(end_date))]


def load_bond_yield(start_date: str, end_date: str) -> pd.Series:
    rf = None
    if hasattr(ak, "bond_zh_us_rate"):
        try:
            raw = ak.bond_zh_us_rate()
            dcol = _find_col(list(raw.columns), ["日期", "date"])
            ycol = _find_col(list(raw.columns), ["中国国债收益率10年", "中国10年", "cn10y", "10y"])
            if dcol and ycol:
                t = raw[[dcol, ycol]].copy()
                t.columns = ["date", "risk_free_rate"]
                t["date"] = pd.to_datetime(t["date"], errors="coerce")
                t["risk_free_rate"] = pd.to_numeric(t["risk_free_rate"], errors="coerce")
                rf = t.dropna().drop_duplicates(subset=["date"]).set_index("date").sort_index()["risk_free_rate"]
        except Exception:
            rf = None

    if rf is None:
        rf = _read_macro_cache("risk_free_rate")
        if rf is None:
            raise RuntimeError("无风险利率在线与缓存均不可用。可手工保存 data/raw/macro/risk_free_rate.csv")
    else:
        _write_macro_cache("risk_free_rate", rf)

    if rf.dropna().median() > 1:
        rf = rf / 100.0
    return rf.loc[(rf.index >= pd.Timestamp(start_date)) & (rf.index <= pd.Timestamp(end_date))]


def merge_macro_data(trading_dates: pd.DatetimeIndex, cpi: pd.Series, rf: pd.Series) -> pd.DataFrame:
    macro = pd.concat([cpi.rename("inflation_rate"), rf.rename("risk_free_rate")], axis=1).sort_index()
    full_index = pd.DatetimeIndex(sorted(set(trading_dates) | set(macro.index)))
    macro = macro.reindex(full_index).sort_index().ffill()
    macro = macro.reindex(trading_dates).ffill().dropna()
    return macro


# ============================================================
# D. 指标层 + 策略层
# ============================================================
def calculate_ma(df: pd.DataFrame, ma_window: int = 120, price_col: str = "adjusted_price") -> pd.DataFrame:
    out = df.copy()
    out["ma"] = out[price_col].rolling(window=ma_window, min_periods=ma_window).mean()
    out["bias"] = out[price_col] / out["ma"]
    return out


def detect_false_signals(df: pd.DataFrame, ma_window: int, buy_th: float, sell_th: float) -> pd.DataFrame:
    raw_x = calculate_ma(df, ma_window=ma_window, price_col="close")
    adj_x = calculate_ma(df, ma_window=ma_window, price_col="adjusted_price")
    out = pd.DataFrame(index=df.index)
    out["false_buy"] = (raw_x["bias"] <= buy_th) & ~(adj_x["bias"] <= buy_th)
    out["false_sell"] = (raw_x["bias"] >= sell_th) & ~(adj_x["bias"] >= sell_th)
    return out


@dataclass
class BacktestResult:
    equity_curve: pd.DataFrame
    trades: pd.DataFrame


def run_backtest_multi(data_dict: Dict[str, pd.DataFrame], ma_window: int, buy_th: float, sell_th: float, initial_cash: float) -> BacktestResult:
    panel = {s: calculate_ma(df, ma_window=ma_window, price_col="adjusted_price") for s, df in data_dict.items()}
    symbols = list(panel.keys())
    dates = panel[symbols[0]].index

    cash = float(initial_cash)
    shares = 0.0
    holding = None
    entry_price = np.nan

    eq_rows, tr_rows = [], []

    for dt in dates:
        # 持仓先判断卖出
        if holding is not None:
            b = panel[holding].at[dt, "bias"]
            p = panel[holding].at[dt, "adjusted_price"]
            if pd.notna(b) and pd.notna(p) and b >= sell_th:
                cash = shares * p
                tr_rows.append({"date": dt, "symbol": holding, "action": "sell", "price": p, "shares": shares, "trade_return": p / entry_price - 1.0})
                holding, shares, entry_price = None, 0.0, np.nan

        # 空仓再选最便宜
        if holding is None:
            candidates = []
            for s in symbols:
                b = panel[s].at[dt, "bias"]
                p = panel[s].at[dt, "adjusted_price"]
                if pd.notna(b) and pd.notna(p) and b <= buy_th:
                    candidates.append((s, b, p))
            if candidates:
                s, b, p = sorted(candidates, key=lambda x: x[1])[0]
                if p > 0:
                    shares = cash / p
                    holding = s
                    entry_price = p
                    cash = 0.0
                    tr_rows.append({"date": dt, "symbol": s, "action": "buy", "price": p, "shares": shares, "trade_return": np.nan})

        equity = cash if holding is None else shares * panel[holding].at[dt, "adjusted_price"]
        eq_rows.append({"date": dt, "equity": equity, "holding": holding if holding else "CASH"})

    eq = pd.DataFrame(eq_rows).set_index("date")
    tr = pd.DataFrame(tr_rows)
    return BacktestResult(equity_curve=eq, trades=tr)


def run_buy_hold(data_dict: Dict[str, pd.DataFrame], symbol: str, initial_cash: float) -> BacktestResult:
    if symbol not in data_dict:
        raise ValueError(f"买入并持有标的 {symbol} 不在 fund_pool 中")
    df = data_dict[symbol]
    px = df["adjusted_price"].copy()
    px = px.ffill().dropna()

    shares = initial_cash / px.iloc[0]
    equity = shares * px
    eq = pd.DataFrame({"equity": equity, "holding": symbol}, index=px.index)
    # buy&hold 默认只有1次买入，不中途卖出
    tr = pd.DataFrame([{ "date": px.index[0], "symbol": symbol, "action": "buy", "price": px.iloc[0], "shares": shares, "trade_return": np.nan }])
    return BacktestResult(equity_curve=eq, trades=tr)


# ============================================================
# E. 绩效评估 + 可视化
# ============================================================
def _annualized_return(nav: pd.Series, annual_days: int = 252) -> float:
    if len(nav) < 2:
        return np.nan
    total = nav.iloc[-1] / nav.iloc[0]
    years = len(nav) / annual_days
    return float(total ** (1.0 / years) - 1.0) if years > 0 else np.nan


def _max_drawdown(nav: pd.Series) -> float:
    return float((nav / nav.cummax() - 1.0).min())


def evaluate_performance(result: BacktestResult, macro_df: pd.DataFrame, tag: str) -> Tuple[pd.DataFrame, dict]:
    perf = result.equity_curve.copy()
    perf["strategy_return"] = perf["equity"].pct_change().fillna(0.0)
    perf["strategy_nav"] = perf["equity"] / perf["equity"].iloc[0]
    perf = perf.join(macro_df, how="left").ffill().dropna()

    daily_infl = (1.0 + perf["inflation_rate"]) ** (1.0 / 252.0) - 1.0
    daily_rf = (1.0 + perf["risk_free_rate"]) ** (1.0 / 252.0) - 1.0

    perf["real_daily_return"] = (1.0 + perf["strategy_return"]) / (1.0 + daily_infl) - 1.0
    perf["excess_daily_return"] = perf["strategy_return"] - daily_rf
    perf["real_nav"] = (1.0 + perf["real_daily_return"]).cumprod()
    perf["excess_nav"] = (1.0 + perf["excess_daily_return"]).cumprod()
    perf["drawdown"] = perf["strategy_nav"] / perf["strategy_nav"].cummax() - 1.0

    ann = _annualized_return(perf["strategy_nav"])
    avg_cpi = float(perf["inflation_rate"].mean())
    avg_rf = float(perf["risk_free_rate"].mean())

    sell_trades = result.trades[result.trades["action"] == "sell"] if not result.trades.empty else pd.DataFrame()
    trade_count = int(len(sell_trades))
    win_rate = float((sell_trades["trade_return"] > 0).mean()) if trade_count > 0 else np.nan

    summary = {
        "tag": tag,
        "total_return": float(perf["strategy_nav"].iloc[-1] - 1.0),
        "annualized_return": ann,
        "max_drawdown": _max_drawdown(perf["strategy_nav"]),
        "trade_count": trade_count,
        "win_rate": win_rate,
        "real_annualized_return": _annualized_return(perf["real_nav"]),
        "excess_annualized_return": _annualized_return(perf["excess_nav"]),
        "beat_cpi": bool(ann > avg_cpi),
        "beat_risk_free": bool(ann > avg_rf),
        "improve_purchasing_power": bool(_annualized_return(perf["real_nav"]) > 0),
    }
    return perf, summary


def plot_performance(perf_multi: pd.DataFrame, perf_hold: pd.DataFrame) -> go.Figure:
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=("MA120轮动 vs 买入并持有", "CPI vs 无风险利率", "轮动策略回撤"),
    )

    fig.add_trace(go.Scatter(x=perf_multi.index, y=perf_multi["strategy_nav"], name="MA120轮动净值"), row=1, col=1)
    fig.add_trace(go.Scatter(x=perf_hold.index, y=perf_hold["strategy_nav"], name="买入并持有净值"), row=1, col=1)
    fig.add_trace(go.Scatter(x=perf_multi.index, y=perf_multi["real_nav"], name="轮动真实净值(扣通胀)"), row=1, col=1)

    fig.add_trace(go.Scatter(x=perf_multi.index, y=perf_multi["inflation_rate"], name="CPI"), row=2, col=1)
    fig.add_trace(go.Scatter(x=perf_multi.index, y=perf_multi["risk_free_rate"], name="10Y国债"), row=2, col=1)

    fig.add_trace(go.Scatter(x=perf_multi.index, y=perf_multi["drawdown"], fill="tozeroy", name="轮动回撤"), row=3, col=1)

    fig.update_layout(height=950, template="plotly_white", hovermode="x unified", title="点金术基金化：MA120轮动与宏观对比")
    fig.update_yaxes(title_text="净值", row=1, col=1)
    fig.update_yaxes(title_text="比率", tickformat=".1%", row=2, col=1)
    fig.update_yaxes(title_text="回撤", tickformat=".1%", row=3, col=1)
    return fig


In [18]:
# ============================================================
# E2. Patch: 防止宏观对齐后 perf 为空导致 IndexError
# ============================================================
def merge_macro_data(trading_dates: pd.DatetimeIndex, cpi: pd.Series, rf: pd.Series) -> pd.DataFrame:
    macro = pd.concat([cpi.rename("inflation_rate"), rf.rename("risk_free_rate")], axis=1).sort_index()

    if macro.empty:
        warnings.warn("宏观数据为空，回退为 0 通胀 / 0 无风险利率。")
        return pd.DataFrame(
            {"inflation_rate": 0.0, "risk_free_rate": 0.0},
            index=pd.DatetimeIndex(trading_dates),
        )

    full_index = pd.DatetimeIndex(sorted(set(trading_dates) | set(macro.index)))
    macro = macro.reindex(full_index).sort_index().ffill().bfill()
    macro = macro.reindex(trading_dates).ffill().bfill()

    # 仍有缺失则回退0，保证可运行
    macro["inflation_rate"] = pd.to_numeric(macro["inflation_rate"], errors="coerce").fillna(0.0)
    macro["risk_free_rate"] = pd.to_numeric(macro["risk_free_rate"], errors="coerce").fillna(0.0)
    return macro


def evaluate_performance(result: BacktestResult, macro_df: pd.DataFrame, tag: str) -> Tuple[pd.DataFrame, dict]:
    perf = result.equity_curve.copy()
    if perf.empty:
        raise RuntimeError(f"{tag}: equity_curve 为空，无法评估绩效")

    perf["strategy_return"] = perf["equity"].pct_change().fillna(0.0)
    perf["strategy_nav"] = perf["equity"] / perf["equity"].iloc[0]

    # 以策略日期为主进行宏观对齐，避免 join+dropna 造成空表
    macro_aligned = macro_df.reindex(perf.index).ffill().bfill() if macro_df is not None else None
    if macro_aligned is None or macro_aligned.empty:
        warnings.warn(f"{tag}: 宏观数据不可用，回退为0")
        macro_aligned = pd.DataFrame({"inflation_rate": 0.0, "risk_free_rate": 0.0}, index=perf.index)

    macro_aligned["inflation_rate"] = pd.to_numeric(macro_aligned["inflation_rate"], errors="coerce").fillna(0.0)
    macro_aligned["risk_free_rate"] = pd.to_numeric(macro_aligned["risk_free_rate"], errors="coerce").fillna(0.0)

    perf = pd.concat([perf, macro_aligned[["inflation_rate", "risk_free_rate"]]], axis=1)

    daily_infl = (1.0 + perf["inflation_rate"]) ** (1.0 / 252.0) - 1.0
    daily_rf = (1.0 + perf["risk_free_rate"]) ** (1.0 / 252.0) - 1.0

    perf["real_daily_return"] = (1.0 + perf["strategy_return"]) / (1.0 + daily_infl) - 1.0
    perf["excess_daily_return"] = perf["strategy_return"] - daily_rf
    perf["real_nav"] = (1.0 + perf["real_daily_return"]).cumprod()
    perf["excess_nav"] = (1.0 + perf["excess_daily_return"]).cumprod()
    perf["drawdown"] = perf["strategy_nav"] / perf["strategy_nav"].cummax() - 1.0

    ann = _annualized_return(perf["strategy_nav"])
    avg_cpi = float(perf["inflation_rate"].mean())
    avg_rf = float(perf["risk_free_rate"].mean())

    sell_trades = result.trades[result.trades["action"] == "sell"] if not result.trades.empty else pd.DataFrame()
    trade_count = int(len(sell_trades))
    win_rate = float((sell_trades["trade_return"] > 0).mean()) if trade_count > 0 else np.nan

    summary = {
        "tag": tag,
        "total_return": float(perf["strategy_nav"].iloc[-1] - 1.0),
        "annualized_return": ann,
        "max_drawdown": _max_drawdown(perf["strategy_nav"]),
        "trade_count": trade_count,
        "win_rate": win_rate,
        "real_annualized_return": _annualized_return(perf["real_nav"]),
        "excess_annualized_return": _annualized_return(perf["excess_nav"]),
        "beat_cpi": bool(ann > avg_cpi),
        "beat_risk_free": bool(ann > avg_rf),
        "improve_purchasing_power": bool(_annualized_return(perf["real_nav"]) > 0),
    }
    return perf, summary


# 运行前检查
print("Patch loaded: merge_macro_data / evaluate_performance 已增强空数据防护")

Patch loaded: merge_macro_data / evaluate_performance 已增强空数据防护


In [19]:
# ============================================================
# G. Patch: 扩展资金池 + 增强交易日志字段
# ============================================================
# 扩展资金池（红利/红利低波相关ETF，注释标明基金）
params["fund_pool"] = [
    "510880",  # 上证红利ETF
    #"159905",  # 深证红利ETF
    "515180",  # 红利低波ETF
    "512890",  # 红利低波ETF
    "515080",  # 中证红利ETF
    "515100",  # 红利质量ETF
    "515300",  # 红利50ETF
    "560020",  # 红利低波100ETF
    "515450",  # 红利ETF
    "512530",  # 红利价值ETF
    "159545",  # 中证红利低波动ETF
    "588000",  # 科创50ETF（用于扩展对比）
    "510300",  # 沪深300ETF（对照）
    "510500",  # 中证500ETF（对照）
    "159915",  # 创业板ETF（对照）
    "159307"  # 博时中证红利低波 100ETF
    "563020"  # 易方达中证红利低波动 ETF
    "510720"  # 红利国企 ETF
    "561580" # 央企红利 ETF
    "513820" # 汇添富中证港股通高股息 ETF（股息率约 7.1%）
    

]


def run_backtest_multi(data_dict: Dict[str, pd.DataFrame], ma_window: int, buy_th: float, sell_th: float, initial_cash: float) -> BacktestResult:
    panel = {s: calculate_ma(df, ma_window=ma_window, price_col="adjusted_price") for s, df in data_dict.items()}
    symbols = list(panel.keys())
    dates = panel[symbols[0]].index

    cash = float(initial_cash)
    shares = 0.0
    holding = None
    entry_price = np.nan

    eq_rows, tr_rows = [], []

    for dt in dates:
        # 持仓先判断卖出
        if holding is not None:
            b = panel[holding].at[dt, "bias"]
            p = panel[holding].at[dt, "adjusted_price"]
            if pd.notna(b) and pd.notna(p) and b >= sell_th:
                cash_before = cash
                cash_after = shares * p
                tr_rows.append(
                    {
                        "date": dt,
                        "action": "sell",
                        "symbol": holding,
                        "price": p,
                        "shares": shares,
                        "cash_before": cash_before,
                        "cash_after": cash_after,
                        "trade_return": p / entry_price - 1.0,
                    }
                )
                cash = cash_after
                holding, shares, entry_price = None, 0.0, np.nan

        # 空仓再选最便宜
        if holding is None:
            candidates = []
            for s in symbols:
                b = panel[s].at[dt, "bias"]
                p = panel[s].at[dt, "adjusted_price"]
                if pd.notna(b) and pd.notna(p) and b <= buy_th:
                    candidates.append((s, b, p))
            if candidates:
                s, b, p = sorted(candidates, key=lambda x: x[1])[0]
                if p > 0:
                    cash_before = cash
                    shares = cash / p
                    holding = s
                    entry_price = p
                    cash = 0.0
                    tr_rows.append(
                        {
                            "date": dt,
                            "action": "buy",
                            "symbol": s,
                            "price": p,
                            "shares": shares,
                            "cash_before": cash_before,
                            "cash_after": cash,
                            "trade_return": np.nan,
                        }
                    )

        equity = cash if holding is None else shares * panel[holding].at[dt, "adjusted_price"]
        eq_rows.append({"date": dt, "equity": equity, "holding": holding if holding else "CASH"})

    eq = pd.DataFrame(eq_rows).set_index("date")
    tr = pd.DataFrame(tr_rows)
    return BacktestResult(equity_curve=eq, trades=tr)


def format_trade_signal_log(trades_df: pd.DataFrame) -> pd.DataFrame:
    if trades_df is None or trades_df.empty:
        return pd.DataFrame(columns=["时间", "操作", "标的", "价格", "份额", "操作前现金", "操作后现金", "单笔收益率"])

    out = trades_df.copy().sort_values("date")
    out["时间"] = pd.to_datetime(out["date"]).dt.strftime("%Y-%m-%d")
    out["操作"] = out["action"].map({"buy": "买入", "sell": "卖出"}).fillna(out["action"])
    out["标的"] = out["symbol"]
    out["价格"] = out["price"].astype(float)
    out["份额"] = out["shares"].astype(float)
    out["操作前现金"] = out["cash_before"].astype(float)
    out["操作后现金"] = out["cash_after"].astype(float)
    out["单笔收益率"] = out["trade_return"].astype(float)

    cols = ["时间", "操作", "标的", "价格", "份额", "操作前现金", "操作后现金", "单笔收益率"]
    return out[cols]


print("Patch loaded: fund_pool expanded + trade signal log enhanced")

Patch loaded: fund_pool expanded + trade signal log enhanced


In [20]:
# ============================================================
# F. 运行主流程（按顺序执行）
# ============================================================
fund_data, load_log = load_fund_data(
    symbols=params["fund_pool"],
    start_date=params["start_date"],
    end_date=params["end_date"],
)

print("基金加载日志（含增量更新与缓存回退状态）")
display(load_log)

# 分红/除息导致的假信号诊断（raw close vs adjusted_price）
false_rows = []
for symbol, df in fund_data.items():
    r = detect_false_signals(
        df,
        ma_window=params["ma_window"],
        buy_th=params["buy_threshold"],
        sell_th=params["sell_threshold"],
    )
    false_rows.append(
        {
            "symbol": symbol,
            "false_buy_count": int(r["false_buy"].sum()),
            "false_sell_count": int(r["false_sell"].sum()),
            "false_total": int(r["false_buy"].sum() + r["false_sell"].sum()),
        }
    )
false_signal_report = pd.DataFrame(false_rows).sort_values("false_total", ascending=False)

print("除息假信号诊断")
display(false_signal_report)

# 策略回测
multi_result = run_backtest_multi(
    data_dict=fund_data,
    ma_window=params["ma_window"],
    buy_th=params["buy_threshold"],
    sell_th=params["sell_threshold"],
    initial_cash=params["initial_cash"],
)

hold_result = run_buy_hold(
    data_dict=fund_data,
    symbol=params["benchmark_hold_symbol"],
    initial_cash=params["initial_cash"],
)

# 宏观数据
cpi = load_cpi_data(params["start_date"], params["end_date"])
rf = load_bond_yield(params["start_date"], params["end_date"])
macro = merge_macro_data(multi_result.equity_curve.index, cpi, rf)

# 绩效评估
perf_multi, summary_multi = evaluate_performance(multi_result, macro, tag="ma120_multi_rotation")
perf_hold, summary_hold = evaluate_performance(hold_result, macro, tag=f"buy_hold_{params['benchmark_hold_symbol']}")

summary_df = pd.DataFrame([summary_multi, summary_hold]).set_index("tag")
print("核心指标对比")
display(summary_df)

verification = pd.DataFrame(
    {
        "check": [
            "1) 全收益口径计算（adjusted_price）",
            "2) 是否检测到除息导致的虚假信号",
            "3) MA120轮动年化是否高于买入并持有",
            "4) MA120轮动是否跑赢CPI",
            "5) MA120轮动是否跑赢无风险利率",
            "6) MA120轮动是否提升真实购买力",
        ],
        "result": [
            True,
            bool(false_signal_report["false_total"].sum() > 0),
            bool(summary_multi["annualized_return"] > summary_hold["annualized_return"]),
            bool(summary_multi["beat_cpi"]),
            bool(summary_multi["beat_risk_free"]),
            bool(summary_multi["improve_purchasing_power"]),
        ],
    }
)

print("最终验证清单")
display(verification)

# 买卖信号日志（按 时间-现金-标的-操作后现金）
trade_log_df = format_trade_signal_log(multi_result.trades)
print("MA120轮动买卖信号日志")
display(trade_log_df)

fig = plot_performance(perf_multi, perf_hold)
fig.show()

基金加载日志（含增量更新与缓存回退状态）


,symbol,status,source,rows,cache_used,updated
0,510880,ok,cache_no_update_needed,2674,True,False
1,515180,ok,cache_no_update_needed,1509,True,False
2,512890,ok,cache_no_update_needed,1732,True,False
3,515080,ok,cache_no_update_needed,1504,True,False
4,515100,ok,cache_no_update_needed,1382,True,False
5,515300,ok,cache_no_update_needed,1570,True,False
6,560020,"failed:akshare 拉取失败: ('Connection aborted.', R...",none,0,False,False
7,515450,"failed:akshare 拉取失败: ('Connection aborted.', R...",none,0,False,False
8,512530,"failed:akshare 拉取失败: ('Connection aborted.', R...",none,0,False,False
9,159545,"failed:akshare 拉取失败: ('Connection aborted.', R...",none,0,False,False


除息假信号诊断


,symbol,false_buy_count,false_sell_count,false_total
0,510880,0,0,0
1,515180,0,0,0
2,512890,0,0,0
3,515080,0,0,0
4,515100,0,0,0
5,515300,0,0,0


C:\Users\HZF\AppData\Local\Temp\ipykernel_28212\3255586344.py:260: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



...

,total_return,annualized_return,max_drawdown,trade_count,win_rate,real_annualized_return,excess_annualized_return,beat_cpi,beat_risk_free,improve_purchasing_power
tag,,,,,,,,,,
ma120_multi_rotation,0.822357,0.058187,-0.221675,5,1.0,0.058187,0.028552,True,True,True
buy_hold_510880,0.627249,0.046954,-0.432802,0,NaN,0.046954,0.017631,True,True,True


最终验证清单


,check,result
0,1) 全收益口径计算（adjusted_price）,True
1,2) 是否检测到除息导致的虚假信号,False
2,3) MA120轮动年化是否高于买入并持有,True
3,4) MA120轮动是否跑赢CPI,True
4,5) MA120轮动是否跑赢无风险利率,True
5,6) MA120轮动是否提升真实购买力,True


MA120轮动买卖信号日志


,时间,操作,标的,价格,份额,操作前现金,操作后现金,单笔收益率
0,2015-09-07,买入,510880,2.871,3.483107e+05,1.000000e+06,0.000000e+00,NaN
1,2016-08-12,卖出,510880,2.951,3.483107e+05,0.000000e+00,1.027865e+06,0.027865
2,2018-06-26,买入,510880,3.256,3.156833e+05,1.027865e+06,0.000000e+00,NaN
3,2019-02-25,卖出,510880,3.476,3.156833e+05,0.000000e+00,1.097315e+06,0.067568
4,2020-02-03,买入,512890,1.047,1.048057e+06,1.097315e+06,0.000000e+00,NaN
5,2020-07-03,卖出,512890,1.223,1.048057e+06,0.000000e+00,1.281773e+06,0.168099
6,2022-03-15,买入,515080,1.427,8.982292e+05,1.281773e+06,0.000000e+00,NaN
7,2023-05-04,卖出,515080,1.739,8.982292e+05,0.000000e+00,1.562021e+06,0.218641
8,2024-09-09,买入,512890,1.932,8.084993e+05,1.562021e+06,0.000000e+00,NaN
9,2024-09-30,卖出,512890,2.254,8.084993e+05,0.000000e+00,1.822357e+06,0.166667
